# Neural Models Laboratory — Submission Notebook

This notebook implements the required XOR laboratory experiments: the Task 1 linear baseline, the 2–2–1 binary model, gradient/backpropagation inspection, zero-initialisation symmetry, the sigmoid/tanh/ReLU activation comparison, and the three-class extension.

**Reproducibility:** seed = 2; Adam learning rate = 0.1; 2000 full-batch steps; CPU PyTorch.

## Task 1 — Problem specification

The input is two binary sensor values, x1 and x2. The output is a binary sensor-disagreement warning:

| x1 | x2 | y |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

The two classes occupy diagonally opposite corners of the square, so one straight decision boundary cannot separate them. A single affine transformation followed by a sigmoid still has a linear decision boundary and therefore cannot represent XOR exactly.

In [ ]:
import torch
import numpy as np
torch.set_num_threads(1)
SEED, LR, STEPS = 2, 0.1, 2000
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([[0.],[1.],[1.],[0.]])
print(X)
print(y.squeeze())

In [ ]:
# Linear-only baseline
import torch.nn as nn
class LinearBinary(nn.Module):
    def __init__(self):
        super().__init__(); self.fc=nn.Linear(2,1)
    def forward(self,x): return self.fc(x)

torch.manual_seed(SEED)
model=LinearBinary(); loss_fn=nn.BCEWithLogitsLoss()
opt=torch.optim.Adam(model.parameters(),lr=LR)
initial=loss_fn(model(X),y).item()
for _ in range(STEPS):
    opt.zero_grad(); loss=loss_fn(model(X),y); loss.backward(); opt.step()
with torch.no_grad():
    final=loss_fn(model(X),y).item()
    p=torch.sigmoid(model(X)).squeeze().numpy()
    pred=(p>=.5).astype(int)
print("Initial loss:",initial)
print("Final loss:",final)
print("Probabilities:",p)
print("Predictions:",pred)

The linear baseline stays near the ambiguous 0.5 probabilities and does not solve XOR, illustrating that the problem is representational rather than merely a lack of parameters.

## Task 2 — Model design

**Architecture:** 2 inputs → 2 hidden units → 1 output.  
**Hidden activation:** sigmoid for the main run; tanh and ReLU later.  
**Output:** one raw logit; sigmoid is applied only when reporting probabilities.  
**Loss:** `BCEWithLogitsLoss`.  
**Optimiser:** Adam.

The hidden nonlinearity is necessary because affine layers compose to another affine map. Sigmoid plus binary cross-entropy is appropriate for a binary yes/no target. Validation checks are: loss reduction, all four correct labels, nonzero gradients, reproducibility, and the expected symmetry behaviour under identical/zero initialisation.

## Task 3 — LLM prompt

Exact prompt used:

> Generate minimal PyTorch code for the following laboratory experiment. Do not change the architecture or task. Use the four XOR examples [[0,0],[0,1],[1,0],[1,1]] with labels [0,1,1,0]. Implement a 2-input -> 2-hidden-unit -> 1-output network, with a nonlinear hidden activation, sigmoid binary prediction, random initialisation, and BCEWithLogitsLoss. Use full-batch training for a few thousand CPU steps. Set a random seed for reproducibility. After training, report the initial and final loss, all four probabilities, thresholded labels, and one first-layer gradient tensor after backward(). Keep the code concise and explain where the forward pass, loss, backward pass, and optimizer update occur. Also make it easy to repeat the run with sigmoid, tanh, and ReLU while changing only the hidden activation.

**Corrections made before execution:** used raw logits with `BCEWithLogitsLoss`; added an explicit gradient-norm check and fixed seed; kept the architecture fixed while changing only activation; added the symmetry and three-class experiments.

In [ ]:
# Final binary XOR implementation
class BinaryNet(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.fc1=nn.Linear(2,2); self.fc2=nn.Linear(2,1); self.activation=activation
    def forward(self,x):
        h=self.activation(self.fc1(x))   # forward pass
        return self.fc2(h)               # raw logit

def run_binary(activation):
    torch.manual_seed(SEED)
    m=BinaryNet(activation); loss_fn=nn.BCEWithLogitsLoss()
    opt=torch.optim.Adam(m.parameters(),lr=LR)
    initial=loss_fn(m(X),y).item()
    early=None
    for step in range(STEPS):
        opt.zero_grad(); logits=m(X); loss=loss_fn(logits,y)
        loss.backward()                  # backpropagation
        if step==9: early=m.fc1.weight.grad.detach().norm().item()
        opt.step()                       # parameter update
    with torch.no_grad():
        final=loss_fn(m(X),y).item()
        p=torch.sigmoid(m(X)).squeeze().numpy()
        pred=(p>=.5).astype(int)
    return initial,final,p,pred,early

result=run_binary(torch.sigmoid)
print("Initial loss:",result[0])
print("Final loss:",result[1])
print("Probabilities:",result[2])
print("Predictions:",result[3])
print("Early ||grad_W1||_2:",result[4])

In [ ]:
# Task 4B — gradient inspection
torch.manual_seed(SEED)
m=BinaryNet(torch.sigmoid); loss_fn=nn.BCEWithLogitsLoss()
loss=loss_fn(m(X),y); loss.backward()
print("Loss:",loss.item())
print("fc1.weight.grad:\n",m.fc1.weight.grad)
print("Gradient norm:",m.fc1.weight.grad.norm().item())

For a mean loss over four examples, the first-layer gradient is the average of the four example-wise gradients. `parameter.grad` represents ∂L/∂W for that parameter tensor.

In [ ]:
# Task 4C — zero-initialisation symmetry
torch.manual_seed(SEED)
sym=BinaryNet(torch.sigmoid)
with torch.no_grad():
    for p in sym.parameters(): p.zero_()
opt=torch.optim.Adam(sym.parameters(),lr=LR)
for step in range(20):
    opt.zero_grad(); loss=loss_fn(sym(X),y); loss.backward()
    if step in [0,1,4,9,19]:
        print("step",step,"loss",loss.item())
        print("hidden weights:\n",sym.fc1.weight)
        print("hidden gradients:\n",sym.fc1.weight.grad)
    opt.step()

The two hidden units remain identical. In the all-zero experiment, the first-layer gradient is exactly zero, so the hidden weights stay zero. More generally, identical hidden units receive identical gradients and cannot spontaneously learn distinct features under deterministic gradient descent.

In [ ]:
# Task 4D — activation comparison
for name,activation in [("Sigmoid",torch.sigmoid),("Tanh",torch.tanh),("ReLU",torch.relu)]:
    r=run_binary(activation)
    print(name, "| final loss =",r[1], "| 4/4 correct =",np.array_equal(r[3],[0,1,1,0]),
          "| early gradient norm =",r[4])

Only the activation changes in the comparison. Different activation derivatives alter gradient magnitudes through the chain rule. Sigmoid and tanh can saturate; ReLU has derivative 0 on its negative branch and 1 on its positive branch. These mechanisms explain the observed differences without implying a universal best activation from four data points.

In [ ]:
# Task 5 — three-class extension
# 0=(0,0), 1=(0,1)/(1,0), 2=(1,1)
yc=torch.tensor([0,1,1,2])

class ThreeClassNet(nn.Module):
    def __init__(self):
        super().__init__(); self.fc1=nn.Linear(2,2); self.fc2=nn.Linear(2,3)
    def forward(self,x): return self.fc2(torch.tanh(self.fc1(x)))

torch.manual_seed(SEED)
m=ThreeClassNet(); ce=nn.CrossEntropyLoss(); opt=torch.optim.Adam(m.parameters(),lr=LR)
for _ in range(STEPS):
    opt.zero_grad(); logits=m(X); loss=ce(logits,yc); loss.backward(); opt.step()
with torch.no_grad():
    logits=m(X); probs=torch.softmax(logits,dim=1); pred=probs.argmax(1)
print("Output weight shape:",tuple(m.fc2.weight.shape))
print("Final loss:",ce(logits,yc).item())
print("Probabilities:\n",probs)
print("Predictions:",pred.tolist())
print("Probability sum for example 2:",probs[1].sum().item())
print("Max change after adding 100 to all logits:",
      (torch.softmax(logits+100,dim=1)-probs).abs().max().item())

For multiclass cross-entropy, the logit gradient is **p − y**. Softmax probabilities sum to one. Adding the same constant to every logit leaves softmax unchanged, which is why subtracting the maximum logit before exponentiation is numerically stable.